# Chapitre 2b : SQL analytique, les 8 questions d'Awa (SM-120 à SM-127)

## À comprendre
Une question métier se traduit d'abord en **définition précise**, puis en requête, puis en **invariant** : une vérité qui doit tenir sur n'importe quel jeu de données (« la somme des régions égale le total »). Si l'invariant casse, c'est un bug.

Les requêtes sont dans `sql/10_questions.sql` (avec leurs commentaires). Chaque cellule ci-dessous affiche la requête puis son résultat.

In [1]:
import re
import duckdb, pandas as pd, psycopg
from IPython.display import Markdown, display
from senmarket_bases.connexions import RACINE, connexion_postgres

S = "principal"
PARQUET = RACINE / "data" / "processed" / "date=2026-10-29"
texte = (RACINE / "sql" / "10_questions.sql").read_text(encoding="utf-8")

# one chunk per ticket: split on the "-- ====" banner that precedes each "-- SM-xxx :" line
TICKETS = {re.match(r"-- (SM-\d+)", m).group(1): m
           for m in re.split(r"(?m)^-- =+\n(?=-- SM-\d+ :)", texte)[1:]}

def instructions(chunk):
    """Split a ticket into statements (a statement ends with a line finishing by ';')."""
    buf, out = [], []
    for ligne in chunk.splitlines():
        buf.append(ligne)
        if ligne.strip().endswith(";") and not ligne.strip().startswith("--"):
            out.append("\n".join(buf)); buf = []
    return out

def executer(stmt):
    code = "\n".join(l for l in stmt.splitlines() if not l.strip().startswith("--"))
    try:
        with connexion_postgres() as conn:
            conn.execute(f"SET search_path TO {S}")
            cur = conn.execute(code)
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]), "PostgreSQL"
    except psycopg.errors.UndefinedTable:
        # navigation is not loaded in PostgreSQL: same SQL, run by DuckDB on the Parquet file
        con = duckdb.connect()
        con.execute(f"CREATE VIEW navigation AS SELECT * FROM read_parquet('{PARQUET}/navigation.parquet')")
        return con.execute(code.rstrip().rstrip(";")).df(), "DuckDB sur le Parquet (navigation absente de PostgreSQL)"

def ticket(num):
    for stmt in instructions(TICKETS[num]):
        display(Markdown("```sql\n" + stmt + "\n```"))
        df, moteur = executer(stmt)
        if moteur != "PostgreSQL":
            print("->", moteur)
        display(df)

## SM-120 : « Combien chaque région nous rapporte-t-elle ? »
**À comprendre.** Chaîne de jointures `lignes -> commandes -> clients`, puis `GROUP BY`. Le CA est la somme de `montant_xof` de toutes les lignes.

In [2]:
ticket("SM-120")

```sql
-- SM-120 : « Combien chaque région nous rapporte-t-elle ? »
-- CA = somme de lignes_commande.montant_xof (toutes les lignes, quel que soit
-- le statut de la commande ou du paiement : même définition qu'au module 0).
-- Chaîne de jointures : lignes_commande -> commandes -> clients.
-- ===========================================================================
SELECT
    cl.region,
    SUM(l.montant_xof)                                            AS ca_xof,
    COUNT(DISTINCT c.commande_id)                                 AS nb_commandes,
    ROUND(100.0 * SUM(l.montant_xof) / SUM(SUM(l.montant_xof)) OVER (), 1) AS part_pct
FROM lignes_commande l
JOIN commandes c  ON c.commande_id = l.commande_id
JOIN clients   cl ON cl.client_id  = c.client_id
GROUP BY cl.region
ORDER BY ca_xof DESC;
```

,region,ca_xof,nb_commandes,part_pct
0,Dakar,65064469,1692,39.9
1,Thiès,31499993,756,19.3
2,Diourbel,13369825,366,8.2
3,Saint-Louis,12059134,306,7.4
4,Kaolack,9602807,234,5.9
5,Louga,5415769,112,3.3
6,Kaffrine,4991685,117,3.1
7,Fatick,4014282,91,2.5
8,Ziguinchor,3603740,97,2.2
9,Kolda,3451297,85,2.1


```sql

-- Invariant 1 : la somme des régions égale le CA total (l'écart doit valoir 0).
WITH par_region AS (
    SELECT cl.region, SUM(l.montant_xof) AS ca_xof
    FROM lignes_commande l
    JOIN commandes c  ON c.commande_id = l.commande_id
    JOIN clients   cl ON cl.client_id  = c.client_id
    GROUP BY cl.region
)
SELECT
    (SELECT SUM(montant_xof) FROM lignes_commande)                    AS ca_total,
    SUM(ca_xof)                                                        AS somme_des_regions,
    (SELECT SUM(montant_xof) FROM lignes_commande) - SUM(ca_xof)       AS ecart
FROM par_region;
```

,ca_total,somme_des_regions,ecart
0,163011763,163011763,0


```sql

-- Invariant 2 (anti-piège) : les jointures ne multiplient ni ne perdent aucune ligne.
SELECT
    (SELECT COUNT(*) FROM lignes_commande) AS lignes_avant,
    COUNT(*)                               AS lignes_apres_jointures
FROM lignes_commande l
JOIN commandes c  ON c.commande_id = l.commande_id
JOIN clients   cl ON cl.client_id  = c.client_id;
```

,lignes_avant,lignes_apres_jointures
0,9995,9995


```sql

-- Contrôle de données : les clients de ville « Inconnue » ont-ils une région cohérente ?
SELECT ville, region, COUNT(*) AS nb_clients
FROM clients
WHERE ville = 'Inconnue' OR region = 'Inconnue'
GROUP BY ville, region
ORDER BY nb_clients DESC;
```

,ville,region,nb_clients
0,Inconnue,Dakar,1


**À lire.** Première table : le CA par région avec sa part. Ensuite : `ecart` doit valoir 0 (la somme des régions égale le total), et `lignes_avant = lignes_apres` prouve que les jointures n'ont multiplié ni perdu aucune ligne (le piège classique).

## SM-121 : « Quels sont les trois produits les plus vendus de chaque catégorie ? »
**À comprendre.** Une **fonction de fenêtre** (`PARTITION BY categorie`) classe sans réduire les lignes, contrairement à `GROUP BY`. `ROW_NUMBER` départage toujours ; `RANK` laisse un trou après une égalité ; `DENSE_RANK` n'en laisse pas.

In [3]:
ticket("SM-121")

```sql
-- SM-121 : « Quels sont les trois produits les plus vendus de chaque catégorie ? »
-- Définition : « le plus vendu » = quantité totale vendue (SUM(quantite)).
-- PARTITION BY categorie : le classement repart de 1 dans chaque catégorie.
-- Deux fenêtres : ROW_NUMBER a besoin d'un départage fixe (produit_id), sinon
-- le résultat pourrait changer d'une exécution à l'autre ; RANK et DENSE_RANK
-- doivent classer SANS départage pour montrer ce qu'elles font des égalités.
-- ===========================================================================
-- Résultat final : au plus 3 produits par catégorie
WITH ventes AS (
    SELECT p.categorie, p.produit_id, p.nom_produit, SUM(l.quantite) AS qte_vendue
    FROM lignes_commande l
    JOIN produits p ON p.produit_id = l.produit_id
    GROUP BY p.categorie, p.produit_id, p.nom_produit
),
classes AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY qte_vendue DESC, produit_id) AS rang
    FROM ventes
)
SELECT categorie, rang, produit_id, nom_produit, qte_vendue
FROM classes
WHERE rang <= 3
ORDER BY categorie, rang;
```

,categorie,rang,produit_id,nom_produit,qte_vendue
0,Alimentation,1,PRO0001,Riz brisé,1729
1,Alimentation,2,PRO0002,Huile végétale,1637
2,Alimentation,3,PRO0003,Sucre,1606
3,Boissons,1,PRO0019,Thé,706
4,Boissons,2,PRO0020,Café,686
5,Boissons,3,PRO0021,Eau minérale,553
6,Électronique,1,PRO0025,Téléphone,568
7,Électronique,2,PRO0027,Chargeur,476
8,Électronique,3,PRO0026,Écouteurs,457
9,Fournitures,1,PRO0035,Stylo,335


```sql

-- Invariant : au plus 3 produits par catégorie (cette requête doit renvoyer 0 ligne)
WITH ventes AS (
    SELECT p.categorie, p.produit_id, SUM(l.quantite) AS qte_vendue
    FROM lignes_commande l JOIN produits p ON p.produit_id = l.produit_id
    GROUP BY p.categorie, p.produit_id
),
classes AS (
    SELECT categorie,
           ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY qte_vendue DESC, produit_id) AS rang
    FROM ventes
)
SELECT categorie, COUNT(*) AS nb
FROM classes WHERE rang <= 3
GROUP BY categorie HAVING COUNT(*) > 3;
```

,categorie,nb


```sql

-- Les égalités : les trois fonctions côte à côte, là où deux produits ont la même quantité
WITH ventes AS (
    SELECT p.categorie, p.produit_id, SUM(l.quantite) AS qte_vendue
    FROM lignes_commande l JOIN produits p ON p.produit_id = l.produit_id
    GROUP BY p.categorie, p.produit_id
),
classes AS (
    SELECT categorie, produit_id, qte_vendue,
           ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY qte_vendue DESC, produit_id) AS row_number,
           RANK()       OVER (PARTITION BY categorie ORDER BY qte_vendue DESC) AS rank,
           DENSE_RANK() OVER (PARTITION BY categorie ORDER BY qte_vendue DESC) AS dense_rank,
           COUNT(*)     OVER (PARTITION BY categorie, qte_vendue)              AS nb_ex_aequo
    FROM ventes
)
SELECT categorie, produit_id, qte_vendue, row_number, rank, dense_rank
FROM classes
WHERE nb_ex_aequo > 1
ORDER BY categorie, qte_vendue DESC, produit_id;
```

,categorie,produit_id,qte_vendue,row_number,rank,dense_rank


```sql

-- ---------------------------------------------------------------------------
-- SM-121, les égalités (variante de mesure) : aucune égalité n'existe sur la
-- quantité vendue (voir plus haut : 0 ligne), ni sur la partition volume. On en a
-- trouvé sur le NOMBRE DE COMMANDES contenant le produit. La définition de SM-121
-- reste « quantité vendue » ; cette mesure sert seulement à montrer comment
-- ROW_NUMBER, RANK et DENSE_RANK traitent une égalité.
-- On affiche toute la catégorie (pas seulement les ex æquo) pour voir le « trou » de RANK.
-- ---------------------------------------------------------------------------
WITH v AS (
    SELECT p.categorie, p.produit_id, p.nom_produit,
           COUNT(DISTINCT l.commande_id) AS nb_commandes
    FROM lignes_commande l
    JOIN produits p ON p.produit_id = l.produit_id
    GROUP BY p.categorie, p.produit_id, p.nom_produit
),
c AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY nb_commandes DESC, produit_id) AS row_number,
           RANK()       OVER (PARTITION BY categorie ORDER BY nb_commandes DESC)             AS rank,
           DENSE_RANK() OVER (PARTITION BY categorie ORDER BY nb_commandes DESC)             AS dense_rank,
           COUNT(*)     OVER (PARTITION BY categorie, nb_commandes)                          AS nb_ex_aequo
    FROM v
),
d AS (   -- une fonction de fenêtre ne s'imbrique pas dans une autre : d'où cette étape
    SELECT *, MAX(nb_ex_aequo) OVER (PARTITION BY categorie) AS egalite_dans_categorie
    FROM c
)
SELECT categorie, produit_id, nom_produit, nb_commandes, row_number, rank, dense_rank
FROM d
WHERE egalite_dans_categorie > 1
ORDER BY categorie, row_number;
```

,categorie,produit_id,nom_produit,nb_commandes,row_number,rank,dense_rank
0,Boissons,PRO0019,Thé,220,1,1,1
1,Boissons,PRO0020,Café,220,2,1,1
2,Boissons,PRO0021,Eau minérale,187,3,3,2
3,Boissons,PRO0022,Jus de fruits,164,4,4,3
4,Fournitures,PRO0034,Cahier,107,1,1,1
5,Fournitures,PRO0035,Stylo,107,2,1,1
6,Fournitures,PRO0036,Cartable,83,3,3,2


**À lire.** Premier résultat : au plus 3 produits par catégorie (la requête d'invariant doit être vide). Le dernier tableau montre une égalité (sur le nombre de commandes) : comparer les trois colonnes de rang.

## SM-122 : « Comment évolue le chiffre d'affaires mois après mois ? »
**À comprendre.** `LAG` va chercher la valeur du mois précédent. Le premier mois n'a pas de variation : c'est `NULL`, pas 0 (0 % voudrait dire « aucun changement », ce qui est faux). La moyenne mobile ne s'affiche qu'avec 3 mois.

In [4]:
ticket("SM-122")

```sql
-- SM-122 : « Comment évolue le CA mois après mois, et quelle est la variation ? »
-- CA = somme de lignes_commande.montant_xof, rattaché au mois de date_commande.
-- LAG : valeur du mois précédent ; NULL pour le premier mois (pas 0 : 0 % voudrait
-- dire « aucun changement », ce qui est faux). Moyenne mobile : mois courant + 2
-- précédents ; NULL tant qu'il n'y a pas 3 mois. nb_jours / premier_jour / dernier_jour
-- repèrent les mois incomplets aux deux extrémités.
-- ===========================================================================
WITH mensuel AS (
    SELECT date_trunc('month', c.date_commande)::date AS mois,
           SUM(l.montant_xof)                          AS ca_xof,
           COUNT(DISTINCT c.date_commande)             AS nb_jours,
           MIN(c.date_commande)                        AS premier_jour,
           MAX(c.date_commande)                        AS dernier_jour
    FROM lignes_commande l
    JOIN commandes c ON c.commande_id = l.commande_id
    GROUP BY 1
)
SELECT mois, ca_xof,
       LAG(ca_xof) OVER (ORDER BY mois)                                        AS ca_mois_precedent,
       ROUND(100.0 * (ca_xof - LAG(ca_xof) OVER (ORDER BY mois))
             / NULLIF(LAG(ca_xof) OVER (ORDER BY mois), 0), 1)                 AS variation_pct,
       CASE WHEN COUNT(*) OVER w = 3 THEN ROUND(AVG(ca_xof) OVER w, 0) END     AS moyenne_mobile_3m,
       nb_jours, premier_jour, dernier_jour
FROM mensuel
WINDOW w AS (ORDER BY mois ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
ORDER BY mois;
```

,mois,ca_xof,ca_mois_precedent,variation_pct,moyenne_mobile_3m,nb_jours,premier_jour,dernier_jour
0,2024-11-01,149870,None,None,None,4,2024-11-11,2024-11-28
1,2024-12-01,127129,149870,-15.2,None,7,2024-12-03,2024-12-23
2,2025-01-01,1247176,127129,881.0,508058,16,2025-01-01,2025-01-31
3,2025-02-01,791530,1247176,-36.5,721945,15,2025-02-04,2025-02-28
4,2025-03-01,2141777,791530,170.6,1393494,24,2025-03-01,2025-03-30
5,2025-04-01,822686,2141777,-61.6,1251998,22,2025-04-01,2025-04-29
6,2025-05-01,1718133,822686,108.8,1560865,27,2025-05-01,2025-05-31
7,2025-06-01,1883168,1718133,9.6,1474662,24,2025-06-01,2025-06-30
8,2025-07-01,4460039,1883168,136.8,2687113,30,2025-07-01,2025-07-31
9,2025-08-01,3747177,4460039,-16.0,3363461,30,2025-08-01,2025-08-31


```sql

-- Invariant 1 : la somme des mois égale le CA total (l'écart doit valoir 0).
WITH mensuel AS (
    SELECT date_trunc('month', c.date_commande)::date AS mois, SUM(l.montant_xof) AS ca_xof
    FROM lignes_commande l JOIN commandes c ON c.commande_id = l.commande_id
    GROUP BY 1
)
SELECT (SELECT SUM(montant_xof) FROM lignes_commande) - SUM(ca_xof) AS ecart FROM mensuel;
```

,ecart
0,0


```sql

-- Invariant 2 : la variation du premier mois est NULL, et elle est la seule (doit afficher 1 | 1).
WITH mensuel AS (
    SELECT date_trunc('month', c.date_commande)::date AS mois, SUM(l.montant_xof) AS ca_xof
    FROM lignes_commande l JOIN commandes c ON c.commande_id = l.commande_id
    GROUP BY 1
),
v AS (
    SELECT mois, ca_xof - LAG(ca_xof) OVER (ORDER BY mois) AS variation FROM mensuel
)
SELECT COUNT(*) FILTER (WHERE variation IS NULL)                              AS nb_null,
       COUNT(*) FILTER (WHERE variation IS NULL AND mois = (SELECT MIN(mois) FROM mensuel)) AS dont_premier_mois
FROM v;
```

,nb_null,dont_premier_mois
0,1,1


**À lire.** Regarder `nb_jours` : un mois incomplet aux extrémités fausse la variation. L'invariant 1 doit valoir 0 ; l'invariant 2 doit afficher `1 | 1`.

## SM-123 : « Quel est notre panier moyen ? »
**À comprendre.** Agrégation **en deux temps** : une ligne par commande avec son panier (somme de ses lignes), puis la moyenne de ces paniers. Moyenner directement les lignes donnerait un autre chiffre.

In [5]:
ticket("SM-123")

```sql
-- SM-123 : « Quel est notre panier moyen ? »
-- Panier d'une commande = somme de ses lignes (jamais la valeur d'une seule ligne).
-- Agrégation en DEUX temps : (1) une ligne par commande avec son panier, (2) la moyenne de ces paniers.
-- Piège évité : joindre commandes et lignes puis faire AVG d'une valeur de commande
-- compterait chaque commande autant de fois qu'elle a de lignes.
-- ===========================================================================
WITH panier AS (
    SELECT commande_id, SUM(montant_xof) AS montant_panier, COUNT(*) AS nb_lignes
    FROM lignes_commande
    GROUP BY commande_id
)
SELECT
    COUNT(*)                                                                      AS nb_commandes_avec_lignes,
    (SELECT COUNT(*) FROM commandes)                                              AS nb_commandes_total,
    ROUND(AVG(montant_panier), 0)                                                 AS panier_moyen_par_commande,
    ROUND((percentile_cont(0.5) WITHIN GROUP (ORDER BY montant_panier))::numeric, 0) AS panier_median,
    (SELECT ROUND(AVG(montant_xof), 0) FROM lignes_commande)                      AS moyenne_par_ligne,
    ROUND(AVG(nb_lignes), 2)                                                      AS lignes_par_commande
FROM panier;
```

,nb_commandes_avec_lignes,nb_commandes_total,panier_moyen_par_commande,panier_median,moyenne_par_ligne,lignes_par_commande
0,4084,4084,39915,18361,16309,2.45


```sql

-- Invariant 1 : la somme des paniers égale le CA total (écart 0).
-- Invariant 2 : panier moyen / moyenne par ligne = nombre moyen de lignes par commande (écart 0).
WITH panier AS (
    SELECT commande_id, SUM(montant_xof) AS montant_panier, COUNT(*) AS nb_lignes
    FROM lignes_commande GROUP BY commande_id
)
SELECT
    (SELECT SUM(montant_xof) FROM lignes_commande) - SUM(montant_panier)                AS ecart_ca,
    ROUND(AVG(montant_panier) / (SELECT AVG(montant_xof) FROM lignes_commande)
          - AVG(nb_lignes), 6)                                                          AS ecart_ratio
FROM panier;
```

,ecart_ca,ecart_ratio
0,0,0.000000


**À lire.** Comparer `panier_moyen_par_commande` et `moyenne_par_ligne` : ils diffèrent car une commande a plusieurs lignes (le rapport des deux vaut le nombre moyen de lignes par commande, `ecart_ratio` = 0).

## SM-124 : « Combien de temps un client met-il à repasser une deuxième commande ? »
**À comprendre.** `ROW_NUMBER` numérote les commandes de chaque client ; le délai est la date de la 2e moins celle de la 1re. Seuls les clients ayant **au moins deux** commandes comptent (les autres n'ont pas de délai).

In [6]:
ticket("SM-124")

```sql
-- SM-124 : « Combien de temps un client met-il à repasser une deuxième commande ? »
-- ROW_NUMBER numérote les commandes de chaque client (date, puis commande_id pour départager).
-- Délai = date de la 2e commande - date de la 1re (en jours).
-- Seuls les clients ayant AU MOINS 2 commandes comptent : les autres n'ont pas de 2e commande
-- (ce n'est pas un délai de 0 jour).
-- ===========================================================================
WITH ordonnees AS (
    SELECT client_id, date_commande,
           ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY date_commande, commande_id) AS rang
    FROM commandes
),
delais AS (
    SELECT client_id,
           MAX(date_commande) FILTER (WHERE rang = 2) - MAX(date_commande) FILTER (WHERE rang = 1) AS jours
    FROM ordonnees
    WHERE rang <= 2
    GROUP BY client_id
    HAVING COUNT(*) = 2
)
SELECT
    COUNT(*)                                                           AS nb_clients_avec_2_commandes,
    (SELECT COUNT(DISTINCT client_id) FROM commandes)                  AS nb_clients_acheteurs,
    ROUND((percentile_cont(0.5) WITHIN GROUP (ORDER BY jours))::numeric, 1) AS delai_median_jours,
    ROUND(AVG(jours), 1)                                               AS delai_moyen_jours,
    MIN(jours)                                                         AS delai_min,
    MAX(jours)                                                         AS delai_max,
    COUNT(*) FILTER (WHERE jours = 0)                                  AS meme_jour
FROM delais;
```

,nb_clients_avec_2_commandes,nb_clients_acheteurs,delai_median_jours,delai_moyen_jours,delai_min,delai_max,meme_jour
0,1182,2341,64.0,105.6,0,628,9


```sql

-- Invariant : chaque client n'apparaît qu'une fois ; le nombre de clients retenus égale
-- le nombre de clients ayant au moins 2 commandes (écart attendu : 0).
SELECT
    (SELECT COUNT(*) FROM (SELECT client_id FROM commandes GROUP BY client_id HAVING COUNT(*) >= 2) t) AS attendu,
    (SELECT COUNT(*) FROM (
        SELECT client_id FROM (
            SELECT client_id, ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY date_commande, commande_id) AS rang
            FROM commandes) o
        WHERE rang = 2) t)                                                                              AS obtenu;
```

,attendu,obtenu
0,1182,1182


**À lire.** Le nombre de clients retenus doit égaler `attendu = obtenu` dans l'invariant. La médiane est plus fiable que la moyenne si quelques délais sont très longs.

## SM-125 : « Quelles commandes n'ont aucun paiement réussi ? »
**À comprendre.** **Anti-jointure** : chercher ce qui manque, avec `NOT EXISTS` ou `LEFT JOIN ... IS NULL`. Les valeurs réelles du statut s'écrivent avec accent (`réussi`) : on les lit avant de filtrer.

In [7]:
ticket("SM-125")

```sql
-- SM-125 : « Quelles commandes n'ont aucun paiement réussi ? Quels paiements ne
-- correspondent à aucune commande ? »
-- Anti-jointure : NOT EXISTS, puis la même chose avec LEFT JOIN ... IS NULL.
-- ===========================================================================
-- Contrôle préalable : les vraies valeurs du statut (les données font foi).
SELECT statut_paiement, COUNT(*) AS nb FROM paiements GROUP BY 1 ORDER BY nb DESC;
```

,statut_paiement,nb
0,réussi,3541
1,échoué,290
2,en_attente,171


```sql

-- Liste 1 : commandes sans aucun paiement réussi (NOT EXISTS), en séparant deux causes
SELECT
    COUNT(*)                                                                         AS commandes_sans_paiement_reussi,
    COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM paiements q WHERE q.commande_id = c.commande_id)) AS dont_aucun_paiement,
    COUNT(*) FILTER (WHERE EXISTS     (SELECT 1 FROM paiements q WHERE q.commande_id = c.commande_id)) AS dont_paiement_non_reussi
FROM commandes c
WHERE NOT EXISTS (
    SELECT 1 FROM paiements p
    WHERE p.commande_id = c.commande_id AND p.statut_paiement = 'réussi'
);
```

,commandes_sans_paiement_reussi,dont_aucun_paiement,dont_paiement_non_reussi
0,543,82,461


```sql

-- Même liste avec LEFT JOIN ... IS NULL. La condition sur le statut va dans le ON (pas dans le WHERE).
SELECT COUNT(*) AS commandes_sans_paiement_reussi_left_join
FROM commandes c
LEFT JOIN paiements p ON p.commande_id = c.commande_id AND p.statut_paiement = 'réussi'
WHERE p.paiement_id IS NULL;
```

,commandes_sans_paiement_reussi_left_join
0,543


... (2 outputs left out)

```sql

-- Liste 2 : paiements qui ne correspondent à aucune commande.
-- Attendu : 0, car la clé étrangère paiements.commande_id l'interdit.
SELECT COUNT(*) AS paiements_sans_commande
FROM paiements p
WHERE NOT EXISTS (SELECT 1 FROM commandes c WHERE c.commande_id = p.commande_id);
```

,paiements_sans_commande
0,0


```sql

-- Croisement avec l'écart relevé au pré-vol : commandes dont le total des paiements
-- (tous statuts) diffère du total des lignes.
WITH l AS (SELECT commande_id, SUM(montant_xof) AS m FROM lignes_commande GROUP BY commande_id),
     p AS (SELECT commande_id, SUM(montant_xof) AS m FROM paiements GROUP BY commande_id)
SELECT
    COUNT(*)                                                    AS commandes_avec_paiement,
    COUNT(*) FILTER (WHERE l.m <> p.m)                          AS ecart_de_montant,
    ROUND(100.0 * COUNT(*) FILTER (WHERE l.m <> p.m) / COUNT(*), 2) AS pct_ecart,
    COUNT(*) FILTER (WHERE l.m <> p.m AND NOT EXISTS (
        SELECT 1 FROM paiements q WHERE q.commande_id = l.commande_id AND q.statut_paiement = 'réussi')) AS dont_sans_paiement_reussi
FROM l JOIN p ON p.commande_id = l.commande_id;
```

,commandes_avec_paiement,ecart_de_montant,pct_ecart,dont_sans_paiement_reussi
0,4002,41,1.02,5


**À lire.** On sépare deux causes : commandes **sans aucun paiement** et commandes dont le paiement est **non réussi**. L'écart entre les deux écritures doit valoir 0, et `paiements_sans_commande` aussi (la clé étrangère l'interdit).

## SM-126 : « Quel petit groupe de clients fait 80 % de notre chiffre d'affaires ? »
**À comprendre.** Une **somme cumulée** en fenêtre (`ROWS UNBOUNDED PRECEDING`) sur les clients triés par CA décroissant ; on s'arrête au premier client qui atteint 80 %.

In [8]:
ticket("SM-126")

```sql
-- SM-126 : « Quel petit groupe de clients fait 80 % de notre chiffre d'affaires ? »
-- Population : les clients ayant au moins une commande (un client sans commande n'a pas de CA).
-- Somme cumulée en fenêtre : ROWS (une ligne à la fois) et pas le cadre par défaut RANGE,
-- qui donnerait le même cumul à deux clients de même CA ; client_id sert de départage.
-- Entiers : ca_cumule * 10 >= ca_total * 8 évite toute comparaison décimale.
-- ===========================================================================
WITH ca_client AS (
    SELECT c.client_id, SUM(l.montant_xof) AS ca
    FROM lignes_commande l
    JOIN commandes c ON c.commande_id = l.commande_id
    GROUP BY c.client_id
),
cumul AS (
    SELECT client_id, ca,
           ROW_NUMBER() OVER (ORDER BY ca DESC, client_id)                     AS rang,
           SUM(ca) OVER (ORDER BY ca DESC, client_id
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)     AS ca_cumule,
           SUM(ca) OVER ()                                                     AS ca_total,
           COUNT(*) OVER ()                                                    AS nb_clients
    FROM ca_client
)
SELECT
    MIN(rang)                                         AS clients_pour_80_pct_du_ca,
    MAX(nb_clients)                                   AS clients_acheteurs,
    ROUND(100.0 * MIN(rang) / MAX(nb_clients), 1)     AS part_des_clients_pct,
    ROUND(100.0 * MIN(ca_cumule) / MAX(ca_total), 1)  AS ca_atteint_pct
FROM cumul
WHERE ca_cumule * 10 >= ca_total * 8;
```

,clients_pour_80_pct_du_ca,clients_acheteurs,part_des_clients_pct,ca_atteint_pct
0,863,2341,36.9,80.0


```sql

-- Invariant : le dernier cumul vaut 100 % du CA total, et la somme des CA par client égale le CA total.
WITH ca_client AS (
    SELECT c.client_id, SUM(l.montant_xof) AS ca
    FROM lignes_commande l JOIN commandes c ON c.commande_id = l.commande_id
    GROUP BY c.client_id
),
cumul AS (
    SELECT ca, SUM(ca) OVER (ORDER BY ca DESC, client_id
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ca_cumule
    FROM ca_client
)
SELECT ROUND(100.0 * MAX(ca_cumule) / (SELECT SUM(montant_xof) FROM lignes_commande), 1) AS dernier_cumul_pct,
       (SELECT SUM(montant_xof) FROM lignes_commande) - SUM(ca)                           AS ecart_ca
FROM cumul;
```

,dernier_cumul_pct,ecart_ca
0,100.0,0


**À lire.** `part_des_clients_pct` dit quelle fraction des clients fait 80 % du CA. L'invariant : le dernier cumul vaut 100 % et l'écart de CA vaut 0.

## SM-127 : « Combien de sessions finissent avec un article au panier mais sans début de paiement ? »
**À comprendre.** On compte **par session** (`GROUP BY session_id`), jamais par événement : une session qui ajoute trois articles ne compte qu'une fois. L'ordre des événements n'est pas pris en compte.

In [9]:
ticket("SM-127")

```sql
-- SM-127 : « Combien de sessions finissent avec un article au panier mais sans début de paiement ? »
-- Types d'événements réels : page_vue, vue_produit, recherche, ajout_panier, debut_paiement.
-- Règle : on compte par session_id seulement. Dans ces données, client_id est attribué événement par
-- événement (plusieurs client_id dans une même session) : une session n'est donc pas rattachée à un client.
-- Une session est « toute anonyme » si aucun de ses événements n'a de client_id. L'ordre des événements
-- n'est pas pris en compte (présence de ajout_panier, absence de debut_paiement).
-- GROUP BY session_id : une session n'apparaît qu'une seule fois, même avec plusieurs ajouts au panier.
-- ===========================================================================
WITH s AS (
    SELECT session_id,
           COUNT(*) FILTER (WHERE type_evenement = 'ajout_panier')    AS nb_panier,
           COUNT(*) FILTER (WHERE type_evenement = 'debut_paiement')  AS nb_paiement,
           COUNT(*) FILTER (WHERE client_id IS NOT NULL)              AS nb_evt_identifies
    FROM navigation
    GROUP BY session_id
)
SELECT
    COUNT(*)                                                                         AS sessions_total,
    COUNT(*) FILTER (WHERE nb_panier > 0)                                            AS sessions_avec_panier,
    COUNT(*) FILTER (WHERE nb_panier > 0 AND nb_paiement = 0)                        AS panier_sans_paiement,
    ROUND(100.0 * COUNT(*) FILTER (WHERE nb_panier > 0 AND nb_paiement = 0)
          / NULLIF(COUNT(*) FILTER (WHERE nb_panier > 0), 0), 1)                     AS pct_abandon,
    COUNT(*) FILTER (WHERE nb_panier > 0 AND nb_paiement = 0 AND nb_evt_identifies = 0) AS dont_toute_anonyme,
    COUNT(*) FILTER (WHERE nb_paiement > 0 AND nb_panier = 0)                        AS paiement_sans_panier
FROM s;
```

,sessions_total,sessions_avec_panier,panier_sans_paiement,pct_abandon,dont_toute_anonyme,paiement_sans_panier
0,6338,1718,1477,86.0,27,687


```sql

-- Invariant : la version anti-jointure (NOT EXISTS) donne le même nombre de sessions.
-- Sans DISTINCT on compterait une ligne par événement ajout_panier : c'est le piège « une session une seule fois ».
SELECT COUNT(*)                     AS lignes_sans_distinct,
       COUNT(DISTINCT n.session_id) AS sessions_distinctes
FROM navigation n
WHERE n.type_evenement = 'ajout_panier'
  AND NOT EXISTS (SELECT 1 FROM navigation q
                  WHERE q.session_id = n.session_id AND q.type_evenement = 'debut_paiement');
```

,lignes_sans_distinct,sessions_distinctes
0,1685,1477


**À lire.** Le dernier tableau montre le piège : `lignes_sans_distinct` (une ligne par ajout au panier) est plus grand que `sessions_distinctes`.

## Question : `NOT EXISTS` et `LEFT JOIN ... IS NULL` donnent-ils le même résultat, et le même plan ?
Résultat identique : déjà prouvé dans SM-125 (écart = 0). Pour le plan, on demande à PostgreSQL ce qu'il compte faire.

In [10]:
A = f"SELECT c.commande_id FROM {S}.commandes c WHERE NOT EXISTS (SELECT 1 FROM {S}.paiements p WHERE p.commande_id = c.commande_id AND p.statut_paiement = 'réussi')"
B = f"SELECT c.commande_id FROM {S}.commandes c LEFT JOIN {S}.paiements p ON p.commande_id = c.commande_id AND p.statut_paiement = 'réussi' WHERE p.paiement_id IS NULL"
with connexion_postgres() as conn:
    for nom, q in [("NOT EXISTS", A), ("LEFT JOIN ... IS NULL", B)]:
        plan = conn.execute(f"EXPLAIN (FORMAT JSON) {q}").fetchone()[0][0]["Plan"]
        print(f"{nom:<24} -> {plan['Node Type']}")

NOT EXISTS               -> Hash Join
LEFT JOIN ... IS NULL    -> Hash Join


**Réponse.** Si les deux lignes affichent le même type de nœud (en général `Hash Anti Join`), PostgreSQL a reconnu que ce sont **la même question** posée de deux façons : il choisit la même stratégie. On choisit alors l'écriture la plus lisible (`NOT EXISTS`, qui dit « n'a aucun »).

## Ce qu'on retient
- Une définition écrite avant le code, un **invariant** vérifié après.
- Après une jointure, on revérifie le total : un montant de commande sommé sur des lignes est compté plusieurs fois.
- `NULL` n'est pas `0`, une anti-jointure cherche l'absence, et on compte par session, pas par événement.